# Implementation Notebook

#### Import Libraries

In [93]:
import pandas as pd
import numpy as np

#### Config Decisions made during EDA

In [64]:
M_SMOOTHING = 25
MIN_CO_RATINGS = 5
MIN_MOVIE_RATES = 5
TOP_N_USER_RECS = 20
TOP_N_SIMILAR = 15
W_COLLAB = 0.75
W_CONTENT = 0.25
POS_RATING_FLOOR = 4.0
DB_PATH = "../backend/movies.db"

#### Load & Integrity

In [65]:
ratings = pd.read_csv("/home/bukunmi/code/givemore/data/ratings.csv")
movies = pd.read_csv("/home/bukunmi/code/givemore/data/movies.csv")
links = pd.read_csv("/home/bukunmi/code/givemore/data/links.csv")

In [66]:
assert set(ratings.columns) == {"userId", "movieId", "rating", "timestamp"}
assert set(movies.columns) == {"movieId", "title", "genres"}
assert set(links.columns) == {"movieId", "imdbId", "tmdbId"}

In [67]:
assert ratings["userId"].nunique() == 610
assert movies["movieId"].nunique() == 9742
assert links["movieId"].nunique() == 9742

In [68]:
assert set(ratings.movieId) - set(movies.movieId) == set()
assert set(movies.movieId) - set(links.movieId) == set()
assert ratings[["userId","movieId"]].duplicated().sum() == 0

#### Clean Movie Metadata

In [69]:
movies["raw_title"] = movies["title"].str.replace(r"\s+", " ", regex=True).str.strip()
movies["Year"] = movies["raw_title"].str.extract(r"\((\d{4})(?:[–-]\d{4})?\)\s*$", expand=False)
movies["title"] = movies["raw_title"].str.replace(r"\s*\(\d{4}(?:[–-]\d{4})?\)\s*$", "", regex=True)

In [70]:
movies.drop(columns=["raw_title"], inplace=True)

In [71]:
assert movies["Year"].isna().sum() == 12

In [72]:
movies["genre_list"] = movies["genres"].str.split("|")
movies["genre_list"] = movies["genre_list"].apply(lambda genres: [g for g in genres if g != "(no genres listed)"])

In [73]:
assert (movies["genres"] == "(no genres listed)").sum() == 34

#### Foundational Statistics

In [74]:
C = ratings["rating"].mean()
movie_stats = ratings.groupby("movieId")["rating"].agg(["count", "mean"])
user_stats = ratings.groupby("userId")["rating"].agg(["count", "mean"])

In [82]:
assert user_stats.shape[0] == 610
assert movie_stats.shape[0] == 9724
assert C == 3.501556983616962

#### Popular Fallback

In [83]:
def weighted_score(v, R, C, m):
    weighted_score = (v / (v + m)) * R + (m / (v + m)) * C
    return weighted_score

In [86]:
weighted_scores = movie_stats.apply(lambda row: weighted_score(row["count"], row["mean"], C, M_SMOOTHING), axis=1).sort_values(ascending=False)

In [90]:
weighted_scores.dtype

dtype('float64')

In [95]:
assert weighted_scores.shape[0] == 9724
assert weighted_scores.index[0] == 318 # Shawshank
assert np.isclose(weighted_scores.iloc[0], 4.361224925702994)